This notebook loads “corrected base”, applys the base mnpass cards, save as scenario, output as cube .net

In [1]:
import os
import sys
import pandas as pd
import numpy as np

from pyproj import CRS
from pathlib import Path

import network_wrangler
from network_wrangler import load_scenario
from network_wrangler import load_roadway
from network_wrangler import load_transit
from network_wrangler import create_scenario
from network_wrangler import Scenario
from network_wrangler.roadway import write_roadway
from network_wrangler.transit import write_transit

from met_council_wrangler import MetCouncil_Parameters
from met_council_wrangler import metcouncil_roadway
from met_council_wrangler import metcouncil_transit

from cube_wrangler import Parameters
from cube_wrangler import util
from cube_wrangler import roadway
from cube_wrangler import StandardTransit

In [2]:
network_wrangler.setup_logging()

In [3]:
%reload_ext autoreload
%autoreload 2

# remote i/o

In [4]:
metcouncil_wrangler_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler")
cube_wrangler_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler")

cc_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\standard_networks")

net_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks")
output_dir2 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\output")

In [5]:
## base mnpass
project_card_dir1 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMnPASS_v1")
## no build cards
project_card_dir2 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\NoBuild_TPP2023_v1")
project_card_dir3 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\NoBuild_MnPASS2023_v1")
## build scenario 
project_card_dir4 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\Build_TPP2050_v1")
project_card_dir5 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\Build_MnPASS2050_v1")
####temp space if needed
# project_card_dir6= os.path.join("C:/project_card_registry/projects/test")

In [ ]:
metcouncil_parameters = MetCouncil_Parameters(
    metcouncil_wrangler_base_dir=metcouncil_wrangler_dir,
    cube_wrangler_base_dir=cube_wrangler_dir
)

# Load Version01

Version 01 has the standard networks with base corrections, rail links and nodes, external stations and connectors

In [ ]:
version_01_scenario = load_scenario(os.path.join(net_dir, 'v01', 'standard_networks', 'v01_scenario.yml'))

# Create Scenario 02

In [ ]:
version_02_scenario = create_scenario(
    base_scenario = version_01_scenario,
    project_card_filepath = project_card_dir1
)

In [ ]:
version_02_scenario.apply_all_projects()

In [ ]:
version_02_scenario.applied_projects

# Save version 02 standard networks

In [ ]:
version_02_scenario.write(
    os.path.join(net_dir, 'v02base', 'standard_networks'),
    name = 'v02base',
    roadway_file_format = "geojson",
    transit_file_format = "txt",
    roadway_write = True,
    transit_write = True,
    projects_write = True,
    overwrite = True,
    roadway_convert_complex_link_properties_to_single_field=True
)

# Make Travel Model Network

### Add centroid and centroid connectors

In [ ]:
r_net = metcouncil_roadway.add_centroid_and_centroid_connector(
    roadway_network = version_02_scenario.road_net,
    parameters = metcouncil_parameters,
    centroid_file = os.path.join(cc_dir, 'centroid_node.pickle'),
    centroid_connector_link_file = os.path.join(cc_dir, 'cc_link.pickle'),
    centroid_connector_shape_file = os.path.join(cc_dir, 'cc_shape.pickle'),
)

In [ ]:
centroids_df = r_net.links_df[r_net.links_df["centroidconnect"] == True].copy()

centroids_df["bike_access"].value_counts()

### Add Rail access and egress links

In [ ]:
r_net = metcouncil_roadway.add_rail_ae_connections(
    r_net,
    metcouncil_parameters,
    exclude_rail_node_id = [417275, 417255]
)

In [ ]:
m_net = metcouncil_roadway.roadway_standard_to_met_council_network(
    r_net,
    metcouncil_parameters    
)

In [ ]:
# check if missing IDs
# centroids does not have osm and shst IDs
# centroid connectors does not have osm and shst IDs

# if node missing shst id
print(m_net.nodes_df.shst_node_id.isnull().sum())
print(m_net.nodes_df.shst_node_id.nunique())

# if node missing model node id
print(m_net.nodes_df.model_node_id.nunique())

# if link missing 
print(m_net.links_df.shstReferenceId.isnull().sum())
print(m_net.links_df.shstReferenceId.nunique())
print(m_net.links_df.model_link_id.nunique())

# if link missing node id
print(m_net.links_df.fromIntersectionId.isnull().sum())
print(m_net.links_df.toIntersectionId.isnull().sum())

In [ ]:
m_net.nodes_df.columns

In [ ]:
m_net.links_df.columns

In [ ]:
#check column datatype .dtypes
m_net.links_df.MNPASS_CODE.dtype

## Populate MnPASS PAY Links


In [20]:
# step 0: make a copy of the metcouncil links dataframe
input_df=(m_net.links_df).copy()

In [21]:
# step 1: select managed lanes with mnpass code
ML_mnpasscode_df = input_df[
    input_df["MNPASS_CODE"].isin(range(1,100)) & 
    (input_df["managed"] == 1) # excludes the reversible section on i-394 since they are not coded as paralel managed lanes
].copy().reset_index(drop = True)


In [ ]:
ML_mnpasscode_df.managed.value_counts(dropna=False)

In [23]:
# step 2: select the ML access connectors
# we know that the B node of the access connector is the A node of the managed lane
acc_con_df = input_df[input_df["B"].isin(ML_mnpasscode_df["A"].unique())].copy().reset_index(drop=True)
acc_con_df = acc_con_df[acc_con_df["managed"] == 0].copy().reset_index(drop=True)

In [ ]:
acc_con_df.model_link_id.nunique()

In [25]:
# step 3: create a dataframe with the connectors and the MNPASS_PAY value
# match access connector with the ML lane based on: B node of the connector = A node of the ML lane
ML_mnpasscode_df = ML_mnpasscode_df[["A", "MNPASS_CODE"]].copy().reset_index(drop=True).rename(columns = {"A": "B", "MNPASS_CODE": "MNPASS_PAY_UPDATE"})
# creates 1-on-1 join between the access connector and the ML lane
join_code_df = ML_mnpasscode_df.merge(acc_con_df[['A','B']].drop_duplicates(), on='B', how='left')

updated_acc_con_df = pd.merge(acc_con_df, join_code_df, how="left", on=["A", "B"])
updated_acc_con_df = updated_acc_con_df[["A", "B", "MNPASS_PAY_UPDATE"]].copy().reset_index(drop=True)

In [26]:
# step 4: merge the MNPASS_PAY_UPDATE values into the original dataframe
output_df = pd.merge(input_df, updated_acc_con_df, how="left", on=["A", "B"])
output_df["MNPASS_PAY"] = np.where(output_df["MNPASS_PAY_UPDATE"].isnull(), output_df["MNPASS_PAY"], output_df["MNPASS_PAY_UPDATE"])
output_df = output_df.drop(columns=["MNPASS_PAY_UPDATE"])
m_net.links_df = output_df.copy()

# Write model network as shapefile

In [ ]:
#out_cols = ['model_link_id', 'id', 'assign_group', 'drive_access', 'roadway_class',
#            'lanes_AM', 'lanes_MD', 'lanes_PM', 'lanes_NT', 'segment_id', 'HOV', 
#            'price_sov_AM', 'geometry', 'managed']

roadway.write_roadway_as_shp(
    roadway_net = m_net,
    parameters = metcouncil_parameters,
    output_link_shp = os.path.join(output_dir2, 'fullnet_v02', 'shapefile', 'links_v02.shp'),
    output_node_shp = os.path.join(output_dir2, 'fullnet_v02', 'shapefile', 'nodes_v02.shp'),
    #link_output_variables = out_cols,
    data_to_csv = False,
    data_to_dbf = True,
    export_drive_only = False, # if user only wants drive links/nodes in the shapefile
)

# Write model network for Cube

In [ ]:
roadway.write_roadway_as_fixedwidth(
    roadway_net = m_net,
    parameters = metcouncil_parameters,
    zones = metcouncil_parameters.zones,
    output_link_txt = os.path.join(output_dir2, 'fullnet_v02', 'links.txt'),
    output_node_txt = os.path.join(output_dir2,  'fullnet_v02','nodes.txt'),
    output_link_header_width_txt = os.path.join(output_dir2,  'fullnet_v02', 'links_header_width.txt'),
    output_node_header_width_txt = os.path.join(output_dir2,  'fullnet_v02','nodes_header_width.txt'),
    output_cube_network_script = os.path.join(output_dir2,  'fullnet_v02',  'make_complete_network_from_fixed_width_file.s'),
)

In [29]:
version_02_scenario.transit_net.road_net = version_02_scenario.road_net
standard_transit_net = StandardTransit.fromTransitNetwork(version_02_scenario.transit_net, parameters=metcouncil_parameters)

In [ ]:
standard_transit_net = metcouncil_transit.transit_standard_to_met_council_transit_network(
    transit_net = standard_transit_net,
    parameters = metcouncil_parameters,
    line_name_xwalk = os.path.join(output_dir2, 'fullnet_v02', 'line_name_xwalk.csv')
) 

In [31]:
standard_transit_net.write_as_cube_lin(outpath = os.path.join(output_dir2, 'fullnet_v02', 'transit.lin'))